# 5G NR PUSCH Tutorial

This notebook provides an introduction to Sionna's [5G New Radio (NR) module](https://nvlabs.github.io/sionna/api/nr.html) and, in particular, the [physical uplink shared channel (PUSCH)](https://nvlabs.github.io/sionna/api/nr.html#pusch). This module provides implementations of a small subset of the physical layer functionalities as described in the 3GPP specifications [38.211](https://portal.3gpp.org/desktopmodules/Specifications/SpecificationDetails.aspx?specificationId=3213), [38.212](https://portal.3gpp.org/desktopmodules/Specifications/SpecificationDetails.aspx?specificationId=3214) and [38.214](https://portal.3gpp.org/desktopmodules/Specifications/SpecificationDetails.aspx?specificationId=3216). 


You will

- Get an understanding of the different components of a PUSCH configuration, such as the carrier, DMRS, and transport block,
- Learn how to rapidly simulate PUSCH transmissions for multiple transmitters,
- Modify the PUSCHReceiver to use a custom MIMO Detector.

## Table of Contents
* [GPU Configuration and Imports](#GPU-Configuration-and-Imports)
* [A "Hello World!" Example](#A-Hello-World-Example)
* [Carrier Configuration](#Carrier-Configuration)
* [Understanding the DMRS Configuration](#Understanding-the-DMRS-Configuration)
    * [Configuring Multiple Layers](#Configuring-Multiple-Layers)
    * [Controlling the Number of DMRS Symbols in a Slot](#Controlling-the-Number-of-DMRS-Symbols-in-a-Slot)
    * [How to control the number of available DMRS ports?](#How-to-control-the-number-of-available-DMRS-ports?)
* [Transport Blocks and MCS](#Transport-Blocks-and-MCS)
* [Looking into the PUSCHTransmitter](#Looking-into-the-PUSCHTransmitter)
* [Components of the PUSCHReceiver](#Components-of-the-PUSCHReceiver)
* [End-to-end PUSCH Simulations](#End-to-end-PUSCH-Simulations)


## GPU Configuration and Imports

In [6]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# tahtan
# Import Sionna
import sys
sys.path.append('../')
import sionna

# try:
#     import sionna
# except ImportError as e:
#     # Install Sionna if package is not already installed
#     import os
#     os.system("pip install sionna")
#     import sionna

import tensorflow as tf
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible results

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import * 
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, array_to_hash, create_timestamped_folders, b2b, f2f, BinarySource
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper


In [7]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
# from bs4 import BeautifulSoup
import pickle
from collections import namedtuple
import json
from tqdm.notebook import tqdm
import itertools
import io

In [8]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2 import service_account

# Load credentials from JSON key
SERVICE_ACCOUNT_FILE = '/workspaces/thanh/thanhnb10-a1453f778e2b.json'  # Update this path
SCOPES = ['https://www.googleapis.com/auth/drive.file']

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
drive_service = build('drive', 'v3', credentials=credentials)

In [9]:
import io
import pandas as pd
from googleapiclient.http import MediaIoBaseUpload

def upload_parquet_to_drive(df, filename, drive_folder_id=None):
    """Save DataFrame as Parquet in memory and upload directly to Google Drive."""
    buffer = io.BytesIO()
    df.to_parquet(buffer, engine="pyarrow")
    buffer.seek(0)

    file_metadata = {"name": filename}
    if drive_folder_id:
        file_metadata["parents"] = [drive_folder_id]

    media = MediaIoBaseUpload(buffer, mimetype="application/octet-stream", resumable=True)
    file = drive_service.files().create(
        body=file_metadata, media_body=media, fields="id"
    ).execute()

    # print(f"Uploaded {filename} to Google Drive (ID: {file.get('id')})")

def upload_pickle_to_drive(data, filename, drive_folder_id=None):
    """Save a pickle object in memory and upload directly to Google Drive."""
    buffer = io.BytesIO()
    pickle.dump(data, buffer)
    buffer.seek(0)

    file_metadata = {"name": filename}
    if drive_folder_id:
        file_metadata["parents"] = [drive_folder_id]

    media = MediaIoBaseUpload(buffer, mimetype="application/octet-stream", resumable=True)
    file = drive_service.files().create(
        body=file_metadata, media_body=media, fields="id"
    ).execute()

    # print(f"Uploaded {filename} to Google Drive (ID: {file.get('id')})")



## A Hello World Example

Let us start with a simple "Hello, World!" example in which we will simulate PUSCH transmissions from a single transmitter to a single receiver over an AWGN channel.

In [10]:
_num_tx = 1
_num_rx = 1
_num_tx_ant = 1
_num_rx_ant = 8
_carrier_frequency = 2.55e9  # Carrier frequency in Hz.
_link_direction = "uplink"

# Configure antenna arrays
_ue_antenna = Antenna(polarization="single",
                polarization_type="V",
                antenna_pattern="38.901",
                carrier_frequency=_carrier_frequency)

_gnb_array = AntennaArray(num_rows=1,
                        num_cols=_num_rx_ant//2,
                        polarization="dual",
                        polarization_type="cross",
                        antenna_pattern="38.901",
                        carrier_frequency=_carrier_frequency)

_channel_model_0 = CDL("C", 150e-9, _carrier_frequency, _ue_antenna, _gnb_array, _link_direction, min_speed=10)

In [11]:
class MyPUSCHConfig(PUSCHConfig):
    def __init__(self):
        super().__init__(
            carrier_config=CarrierConfig(
                n_cell_id=0,
                cyclic_prefix="normal",
                subcarrier_spacing=30,
                n_size_grid=273,
                n_start_grid=0,
                slot_number=4,
                frame_number=0
            ),
            pusch_dmrs_config=PUSCHDMRSConfig(
                config_type=1,
                length=1,
                additional_position=1,
                dmrs_port_set=[0],
                n_id=0,
                n_scid=0,
                num_cdm_groups_without_data=2,
                type_a_position=2
            ),
            tb_config=TBConfig(
                channel_type='PUSCH',
                n_id=0,
                mcs_table=1,
                mcs_index=9
            ),
            mapping_type='A',
            n_size_bwp=273,
            n_start_bwp=0,
            num_layers=1,
            num_antenna_ports=1,
            precoding='non-codebook',
            tpmi=0,
            transform_precoding=False,
            n_rnti=2008,
            symbol_allocation=[0,14]
        )

_pusch_config_0 = MyPUSCHConfig()
_pusch_config_0.show()

_tb_size = _pusch_config_0.tb_size
_num_coded_bits = _pusch_config_0.num_coded_bits
_target_coderate = _pusch_config_0.tb.target_coderate
_num_bits_per_symbol = _pusch_config_0.tb.num_bits_per_symbol
_num_layers = _pusch_config_0.num_layers
_n_rnti = _pusch_config_0.n_rnti
_n_id = _pusch_config_0.tb.n_id

_dmrs_length = _pusch_config_0.dmrs.length
_dmrs_additional_position = _pusch_config_0.dmrs.additional_position
_num_cdm_groups_without_data = _pusch_config_0.dmrs.num_cdm_groups_without_data
_n_scid = _pusch_config_0.dmrs.n_scid
_n_id_n_scid = _pusch_config_0.dmrs.n_id[0]


Carrier Configuration
cyclic_prefix : normal
cyclic_prefix_length : 2.3437500000000002e-06
frame_duration : 0.01
frame_number : 0
kappa : 64.0
mu : 1
n_cell_id : 0
n_size_grid : 273
n_start_grid : 0
num_slots_per_frame : 20
num_slots_per_subframe : 2
num_symbols_per_slot : 14
slot_number : 4
sub_frame_duration : 0.001
subcarrier_spacing : 30
t_c : 5.086263020833334e-10
t_s : 3.2552083333333335e-08

PUSCH Configuration
dmrs_grid : shape (1, 3276, 14)
dmrs_grid_precoded : shape ()
dmrs_mask : shape (3276, 14)
dmrs_symbol_indices : [2, 11]
frequency_hopping : neither
l : [2, 11]
l_0 : 2
l_bar : [2, 11]
l_d : 14
l_prime : [0]
l_ref : 0
mapping_type : A
n : shape (819,)
n_rnti : 2008
n_size_bwp : 273
n_start_bwp : 0
num_antenna_ports : 1
num_coded_bits : 78624
num_layers : 1
num_ov : 0
num_res_per_prb : 144
num_resource_blocks : 273
num_subcarriers : 3276
precoding : non-codebook
precoding_matrix : None
symbol_allocation : [0, 14]
tb_size : 52224
tpmi : 0
transform_precoding : False

PUSCH 

In [12]:
# _binary_source = BinarySource(dtype=tf.float32)

In [13]:
# _tb_encoder = TBEncoder(
#                 target_tb_size=_tb_size,
#                 num_coded_bits=_num_coded_bits,
#                 target_coderate=_target_coderate,
#                 num_bits_per_symbol=_num_bits_per_symbol,
#                 num_layers=_num_layers,
#                 n_rnti=_n_rnti,
#                 n_id=_n_id,
#                 channel_type="PUSCH", # PUSCHTransmitter
#                 codeword_index=0, # not supported for PUSCH
#                 use_scrambler=True,
#                 verbose=False,
#                 output_dtype=tf.float32)


In [14]:
# _mapper = Mapper("qam", _num_bits_per_symbol, dtype=tf.complex64)

In [15]:
# _layer_mapper = LayerMapper(num_layers=_num_layers, dtype=tf.complex64)

In [16]:
# _pilot_pattern = PUSCHPilotPattern([_pusch_config_0],
#                                         dtype=tf.complex64)
# _pilot_pattern.show();

In [17]:
# _mu = 1
# _num_ofdm_symbols = 14
# _fft_size = 4096
# _cyclic_prefix_length = 288
# _subcarrier_spacing = 30e3
# _num_guard_subcarriers = (410, 410)
# _num_slots_per_frame = 20

# # Define the resource grid.
# _resource_grid = ResourceGrid(
#     num_ofdm_symbols=_num_ofdm_symbols,
#     fft_size=_fft_size,
#     subcarrier_spacing=_subcarrier_spacing,
#     num_tx=1,
#     num_streams_per_tx=1,
#     cyclic_prefix_length=_cyclic_prefix_length,
#     num_guard_carriers=_num_guard_subcarriers,
#     dc_null=False,
#     # pilot_pattern=_pilot_pattern,
#     dtype=tf.complex64
# )
# _resource_grid.show();

In [18]:
# _pusch_config_0.dmrs_grid

In [19]:
# _pusch_config_1 = _pusch_config_0.clone()
# _pusch_config_1.tb.mcs_index = 6
# _pusch_config_1.carrier.slot_number = 5
# _pusch_config_1.dmrs_grid

In [20]:
# _resource_grid_mapper = ResourceGridMapper(_resource_grid, dtype=tf.complex64)

In [21]:
# batch_size = 1
# b = _binary_source([batch_size, _num_tx, _tb_size])

# # Encode transport block
# # if 'tbe' in ret:
# #     c, tbe = _tb_encoder(b, ret=['u', 'u_crc', 'u_cb', 'u_cb_crc', 'c_cb', 'c', 'c_scr' , 'c_tb'])
# c = _tb_encoder(b)

# # Map to constellations
# x_map = _mapper(c)

# # Map to layers
# x_layer = _layer_mapper(x_map)

# # Apply resource grid mapping
# _resource_grid_mapper._resource_grid.pilot_pattern = _pilot_pattern
# x_grid = _resource_grid_mapper(x_layer)

# x = x_grid

In [22]:
# _resource_grid_0 = ResourceGrid(
#     num_ofdm_symbols=_num_ofdm_symbols,
#     fft_size=_fft_size,
#     subcarrier_spacing=_subcarrier_spacing,
#     num_tx=1,
#     num_streams_per_tx=1,
#     cyclic_prefix_length=_cyclic_prefix_length,
#     num_guard_carriers=_num_guard_subcarriers,
#     dc_null=False,
#     dtype=tf.complex64
# )
# _channel_0 = OFDMChannel(_channel_model_0, _resource_grid_0, return_channel=True)

In [23]:
# no = 10.
# y,h = _channel_0([x, no])

In [24]:
# # _ls_est = LSChannelEstimator(_resource_grid, interpolation_type="nn")
# _channel_estimator = PUSCHLSChannelEstimator(
#                 _resource_grid,
#                 _dmrs_length,
#                 _dmrs_additional_position,
#                 _num_cdm_groups_without_data,
#                 interpolation_type='lin',
#                 dtype=tf.complex64)

# _removed_null_subc = RemoveNulledSubcarriers(_resource_grid)

In [25]:
# _rx_tx_association = np.ones([1, _num_tx], bool)
# _stream_management = StreamManagement(_rx_tx_association, _num_layers)
# _mimo_detector = LinearDetector("lmmse", "bit", "maxlog", _resource_grid,
#                                 _stream_management, "qam", _num_bits_per_symbol, dtype=tf.complex64)

In [26]:
# _layer_demapper = LayerDemapper(_layer_mapper, num_bits_per_symbol=_num_bits_per_symbol)
            

In [27]:
# _tb_decoder = TBDecoder(_tb_encoder, output_dtype=tf.float32)

In [28]:
# h_hat,err_var = _channel_estimator([y, no])
# llr_det = _mimo_detector([y, h_hat, err_var, no])
# llr_layer = _layer_demapper(llr_det)
# b_hat, tb_crc_status = _tb_decoder(llr_layer)
# compute_ber(b, b_hat)

In [29]:
# h_hat = _removed_null_subc(h) # Extract non-null subcarriers
# err_var = 0.0
# llr_det = _mimo_detector([y, h_hat, err_var, no])
# llr_layer = _layer_demapper(llr_det)
# b_hat, tb_crc_status = _tb_decoder(llr_layer)
# compute_ber(b, b_hat)

In [30]:
"""
frame_number, n_cell_id: No change
n_rnti, tb.n_id: Change in tb_encoder, but not in dmrs (same c -> same x)
slot_number, dmrs.n_id, dmrs.n_scid, mcs: No change in tb_encoder (same b -> same c), but in dmrs
"""

'\nframe_number, n_cell_id: No change\nn_rnti, tb.n_id: Change in tb_encoder, but not in dmrs (same c -> same x)\nslot_number, dmrs.n_id, dmrs.n_scid, mcs: No change in tb_encoder (same b -> same c), but in dmrs\n'

In [31]:
mcs_params = {
    'tb_size': [],
    'num_coded_bits': [],
    'target_coderate': [],
    'num_bits_per_symbol': []
}
for mcs in range(0,10):
    pusch_config_i = _pusch_config_0.clone()
    pusch_config_i.tb.mcs_index = mcs
    mcs_params['tb_size'].append(pusch_config_i.tb_size)
    mcs_params['num_coded_bits'].append(pusch_config_i.num_coded_bits)
    mcs_params['target_coderate'].append(pusch_config_i.tb.target_coderate)
    mcs_params['num_bits_per_symbol'].append(pusch_config_i.tb.num_bits_per_symbol)

In [32]:
slot_params = {
    'pilots': []
}
pusch_slots_in_frame = [4,5,14,15]
num_pusch_slots_in_frame = len(pusch_slots_in_frame)
for slot_index in pusch_slots_in_frame:
    pusch_config_i = _pusch_config_0.clone()
    pusch_config_i.carrier.slot_number = slot_index
    pilot_pattern_i = PUSCHPilotPattern([pusch_config_i], dtype=tf.complex64)
    slot_params['pilots'].append(pilot_pattern_i.pilots)

In [33]:
PuschRecord = namedtuple("PuschRecord", [
    "nSFN", "nSlot", "nPDU", "nGroup", "nUlsch", "nUlcch", "nRachPresent",
    "nRNTI", "nUEId", "nBWPSize", "nBWPStart", "nSubcSpacing", "nCpType", "nULType",
    "nMcsTable", "nMCS", "nTransPrecode", "nTransmissionScheme", "nNrOfLayers",
    "nPortIndices", "nNid", "nSCID", "nNIDnSCID", "nNrOfAntennaPorts",
    "nVRBtoPRB", "nPMI", "nStartSymbolIndex", "nNrOfSymbols", "nResourceAllocType",
    "nRBStart", "nRBSize", "nTBSize", "nRV", "nHARQID", "nNDI", "nMappingType",
    "nDMRSConfigType", "nNrOfCDMs", "nNrOfDMRSSymbols", "nDMRSAddPos",
    "nPTRSPresent", "nAck", "nAlphaScaling", "nBetaOffsetACKIndex", "nCsiPart1",
    "nBetaOffsetCsiPart1Index", "nCsiPart2", "nBetaOffsetCsiPart2Index",
    "nTpPi2BPSK", "nTPPuschID", "nRxRUIdx", "nUE", "nPduIdx",

    # New fields for channel and filenames
    "Channel_model", "Speed", "Delay_spread", "Esno_db",
    "Data_filename","Data_dirname"
    ]
)

def save_pickle(data, filename):
    """Saves data to a pickle file."""
    with open(filename, "wb") as f:
        pickle.dump(data, f)

In [34]:
_binary_source = BinarySource(dtype=tf.float32)

_mapper = Mapper("qam", _num_bits_per_symbol, dtype=tf.complex64)

_layer_mapper = LayerMapper(num_layers=_num_layers, dtype=tf.complex64)

_pilot_pattern = PUSCHPilotPattern([_pusch_config_0],
                                        dtype=tf.complex64)
_mu = 1
_num_ofdm_symbols = 14
_fft_size = 4096
_cyclic_prefix_length = 288
_subcarrier_spacing = 30e3
_num_guard_subcarriers = (410, 410)
_num_slots_per_frame = 20

# Define the resource grid.
_resource_grid = ResourceGrid(
    num_ofdm_symbols=_num_ofdm_symbols,
    fft_size=_fft_size,
    subcarrier_spacing=_subcarrier_spacing,
    num_tx=1,
    num_streams_per_tx=1,
    cyclic_prefix_length=_cyclic_prefix_length,
    num_guard_carriers=_num_guard_subcarriers,
    dc_null=False,
    pilot_pattern=_pilot_pattern,
    dtype=tf.complex64
)
_resource_grid_mapper = ResourceGridMapper(_resource_grid, dtype=tf.complex64)

_channel_estimator = PUSCHLSChannelEstimator(
                _resource_grid,
                _dmrs_length,
                _dmrs_additional_position,
                _num_cdm_groups_without_data,
                interpolation_type='lin',
                dtype=tf.complex64)

_rx_tx_association = np.ones([_num_rx, _num_tx], bool)
_stream_management = StreamManagement(_rx_tx_association, _num_layers)
_mimo_detector = LinearDetector("lmmse", "bit", "maxlog", _resource_grid,
                                _stream_management, "qam", _num_bits_per_symbol, dtype=tf.complex64)

def generate_data(name,
        data_dir,
        mcss,
        cdl_models,
        speeds,
        delay_spreads,
        esno_dbs,
        samples_per_case=1):
    
    pusch_records=[]
    parquet_dir = f'{data_dir}/parquet'

    pickle_dir = f'{data_dir}/pickle/{name}'
    os.makedirs(pickle_dir, exist_ok=True)

    total_iterations = len(mcss) * len(cdl_models) * len(speeds) * len(delay_spreads) * len(esno_dbs) * samples_per_case
    with tqdm(total=total_iterations, desc="Generating Data") as pbar:
        for mcs in mcss:
            tb_size = mcs_params['tb_size'][mcs]
            num_coded_bits = mcs_params['num_coded_bits'][mcs]
            target_coderate = mcs_params['target_coderate'][mcs]
            num_bits_per_symbol = mcs_params['num_bits_per_symbol'][mcs]
            tb_encoder = TBEncoder(
                        target_tb_size=tb_size,
                        num_coded_bits=num_coded_bits,
                        target_coderate=target_coderate,
                        num_bits_per_symbol=num_bits_per_symbol,
                        num_layers=_num_layers,
                        n_rnti=_n_rnti,
                        n_id=_n_id,
                        channel_type="PUSCH",
                        codeword_index=0,
                        use_scrambler=True,
                        verbose=False,
                        output_dtype=tf.float32)
            
            for cdl_model, speed, delay_spread in itertools.product(cdl_models, speeds, delay_spreads):
                channel_model_i = CDL(cdl_model, delay_spread, _carrier_frequency, _ue_antenna, _gnb_array, _link_direction, min_speed=speed)
                channel_i = OFDMChannel(channel_model_i, _resource_grid, return_channel=True)
                
                for esno_db in esno_dbs:
                    No = pow(10., -esno_db / 10.)
                    
                    for sample_num in range(samples_per_case):
                        status_str = f"(MCS {mcs} | CDL-{cdl_model} | {speed} m/s | {delay_spread} s) | {esno_db} dB | Sample {sample_num+1}/{samples_per_case}"
                        pbar.set_description(status_str)
                        
                        slot_num = pusch_slots_in_frame[sample_num % num_pusch_slots_in_frame]
                        _resource_grid_mapper._resource_grid.pilot_pattern.pilots = slot_params['pilots'][sample_num % num_pusch_slots_in_frame]

                        b = _binary_source([1, _num_tx, tb_size])
                        c = tb_encoder(b)
                        x_map = _mapper(c)
                        x_layer = _layer_mapper(x_map)
                        x = _resource_grid_mapper(x_layer)

                        y, h = channel_i([x, No])

                        timestamp = datetime.now().strftime("%Y%m%d%H%M%S%f")
                        save_pickle(b, f'{pickle_dir}/{timestamp}.b.pkl')
                        save_pickle(c, f'{pickle_dir}/{timestamp}.c.pkl')
                        save_pickle(y, f'{pickle_dir}/{timestamp}.y.pkl')

                        pusch_records.append(PuschRecord(
                                            nSFN=(sample_num // num_pusch_slots_in_frame) % 1023,
                                            nSlot=slot_num,
                                            nPDU=1,
                                            nGroup=1,
                                            nUlsch=1,
                                            nUlcch=0,
                                            nRachPresent=0,
                                            nRNTI=_n_rnti,
                                            nUEId=0,
                                            nBWPSize=273,
                                            nBWPStart=0,
                                            nSubcSpacing=_mu,
                                            nCpType=0,
                                            nULType=0,
                                            nMcsTable=0,
                                            nMCS=mcs,
                                            nTransPrecode=0,
                                            nTransmissionScheme=0,
                                            nNrOfLayers=_num_layers,
                                            nPortIndices=[0],
                                            nNid=_n_id,
                                            nSCID=_n_scid,
                                            nNIDnSCID=_n_id_n_scid,
                                            nNrOfAntennaPorts=_num_rx_ant,
                                            nVRBtoPRB=0,
                                            nPMI=0,
                                            nStartSymbolIndex=0,
                                            nNrOfSymbols=14,
                                            nResourceAllocType=1,
                                            nRBStart=0,
                                            nRBSize=273,
                                            nTBSize=(tb_size/8).__ceil__(),
                                            nRV=0,
                                            nHARQID=sample_num % 16,
                                            nNDI=1,
                                            nMappingType=0,
                                            nDMRSConfigType=0,
                                            nNrOfCDMs=_num_cdm_groups_without_data,
                                            nNrOfDMRSSymbols=_dmrs_length,
                                            nDMRSAddPos=_dmrs_additional_position,
                                            nPTRSPresent=0,
                                            nAck=0,
                                            nAlphaScaling=0,
                                            nBetaOffsetACKIndex=0,
                                            nCsiPart1=0,
                                            nBetaOffsetCsiPart1Index=0,
                                            nCsiPart2=0,
                                            nBetaOffsetCsiPart2Index=0,
                                            nTpPi2BPSK=0,
                                            nTPPuschID=0,
                                            nRxRUIdx=np.arange(0, _num_rx_ant),
                                            nUE=1,
                                            nPduIdx=[0],
                                            Channel_model=f"CDL-{cdl_model}",
                                            Speed=speed,
                                            Delay_spread=delay_spread,
                                            Esno_db=esno_db,
                                            Data_filename=timestamp,
                                            Data_dirname=name
                                        )
                                    )
                        pbar.update(1)  # Increment progress

                    df = pd.DataFrame.from_records(pusch_records, columns=PuschRecord._fields)
                    df.to_parquet(f'{parquet_dir}/{name}.parquet', engine="pyarrow")
   

In [37]:
samples_per_case = 1

data_dir = '../Dataset/tmp'
os.makedirs(data_dir,exist_ok=True)
os.makedirs(f'{data_dir}/pickle',exist_ok=True)
os.makedirs(f'{data_dir}/parquet',exist_ok=True)

# mcss = [9,7,5,3]

# cdl_models = ["A", "C"]
# speeds = [1, 10]
# delay_spreads = [10e-9,150e-9]

# esno_dbs = [i for i in range(-5,11,0.5)]

name = datetime.now().strftime("%Y%m%d%H%M%S%f")
# generate_data(name=name,
#               data_dir=data_dir,
#               mcss=[9], 
#               cdl_models=['A'],
#               speeds=[0, 3, 10],
#               delay_spreads=[150e-9],
#               esno_dbs=[i for i in np.arange(-3,6.1,1)],
#               samples_per_case=1)

generate_data(name=name,
              data_dir=data_dir,
              mcss=[3], 
              cdl_models=['A'],
              speeds=[0],
              delay_spreads=[150e-9],
              esno_dbs=[-12.],
              samples_per_case=40)

Generating Data:   0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
drive_parquet_folder_id = "1p9e9tw9iFrXiaFFh_TP-cuaHwxwLYMYb"
drive_pickle_folder_id = "1_kx1wUO8eBe7tltrYQhWWNFhS8dbfDFW"

def create_drive_folder(folder_name, parent_folder_id=None):
    """Creates a folder in Google Drive and returns the folder ID."""
    folder_metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder",
    }
    if parent_folder_id:
        folder_metadata["parents"] = [parent_folder_id]

    folder = drive_service.files().create(body=folder_metadata, fields="id").execute()
    print(f'Folder "{folder_name}" created with ID: {folder["id"]}')
    return folder["id"]

def upload_files_to_drive(data_dir, name, folder_ids):
    files_to_upload = [f'{data_dir}/parquet/{name}.parquet']
    for (root, _, file) in os.walk(f'{data_dir}/pickle/{name}'):
        for f in file:
            files_to_upload.append(os.path.join(root, f))

    pickle_subfolder_id = create_drive_folder(name, folder_ids[1])
    """Uploads multiple files sequentially but with optimizations."""
    for file_path in tqdm(files_to_upload, desc="Uploading Files"):
        file_name = file_path.split("/")[-1]
        folder_id = folder_ids[0] if ".parquet" in file_name else pickle_subfolder_id

        file_metadata = {"name": file_name, "parents": [folder_id]}
        media = MediaFileUpload(file_path, resumable=True)  # Enables resumable uploads

        try:
            drive_service.files().create(body=file_metadata, media_body=media, fields="id").execute()
        except Exception as e:
            print(f"Error uploading {file_name}: {e}")

    # if os.path.exists(data_dir): os.rename(data_dir, '../Dataset/'+datetime.now().strftime('%Y%m%d%H%M'))
# Example Usage
data_dir = '../Dataset/tmp'
upload_filename = name
upload_files_to_drive(data_dir, upload_filename, [drive_parquet_folder_id, drive_pickle_folder_id])


NameError: name 'dadada' is not defined

Hopefully you have enjoyed this tutorial on Sionna's 5G NR PUSCH module!

Please have a look at the [API documentation](https://nvlabs.github.io/sionna/api/sionna.html) of the various components or the other available [tutorials](https://nvlabs.github.io/sionna/tutorials.html) to learn more.